# Customer Operations Analytics & Workflow Automation

In [1]:
import pandas as pd
import numpy as np


In [2]:
customers = pd.read_csv("../data/raw/customers_raw.csv")
campaigns = pd.read_csv("../data/raw/campaigns_raw.csv")
responses = pd.read_csv("../data/raw/responses_raw.csv")
support_tickets = pd.read_csv("../data/raw/support_tickets_raw.csv")
transactions = pd.read_csv("../data/raw/transactions_raw.csv")


In [3]:
datasets = {
    "Customers": customers,
    "Campaigns": campaigns,
    "Responses": responses,
    "Support Tickets": support_tickets,
    "Transactions": transactions
}

overview = []

for name, df in datasets.items():
    overview.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Cells": df.isna().sum().sum()
    })

overview_df = pd.DataFrame(overview)

overview_df

,Dataset,Rows,Columns,Duplicate Rows,Missing Cells
0,Customers,5060,7,40,267
1,Campaigns,41,8,1,2
2,Responses,15075,6,75,70
3,Support Tickets,8050,8,50,7079
4,Transactions,12060,7,60,155


In [4]:
customers.head()
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 5060 entries, 0 to 5059
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       5060 non-null   str    
 1   name              5060 non-null   str    
 2   email             5020 non-null   str    
 3   phone             4894 non-null   float64
 4   city              4999 non-null   str    
 5   signup_date       5060 non-null   str    
 6   customer_segment  5060 non-null   str    
dtypes: float64(1), str(6)
memory usage: 276.8 KB


In [5]:
campaigns.head()
campaigns.info()

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   campaign_id      41 non-null     str    
 1   campaign_name    41 non-null     str    
 2   channel          41 non-null     str    
 3   campaign_date    41 non-null     str    
 4   budget           39 non-null     float64
 5   objective        41 non-null     str    
 6   status           41 non-null     str    
 7   target_audience  41 non-null     str    
dtypes: float64(1), str(7)
memory usage: 2.7 KB


In [6]:
responses.head()
responses.info()

<class 'pandas.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   response_id    15075 non-null  str  
 1   customer_id    15025 non-null  str  
 2   campaign_id    15055 non-null  str  
 3   response_date  15075 non-null  str  
 4   response_type  15075 non-null  str  
 5   converted      15075 non-null  int64
dtypes: int64(1), str(5)
memory usage: 706.8 KB


In [7]:
support_tickets.head()
support_tickets.info()

<class 'pandas.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ticket_id           8050 non-null   str    
 1   customer_id         8050 non-null   str    
 2   created_date        8050 non-null   str    
 3   category            7980 non-null   str    
 4   priority            8050 non-null   str    
 5   status              8050 non-null   str    
 6   resolution_date     4544 non-null   str    
 7   satisfaction_score  4547 non-null   float64
dtypes: float64(1), str(7)
memory usage: 503.3 KB


In [8]:
transactions.head()
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 12060 entries, 0 to 12059
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    12060 non-null  str    
 1   customer_id       12060 non-null  str    
 2   transaction_date  12060 non-null  str    
 3   product           12035 non-null  str    
 4   amount            12060 non-null  float64
 5   payment_method    11930 non-null  str    
 6   status            12060 non-null  str    
dtypes: float64(1), str(6)
memory usage: 659.7 KB


# Temporal Relationship Validation
Check whether operational events occur on or after the relevant customer signup and campaign dates.

In [9]:
customer_signup = pd.to_datetime(
    customers.drop_duplicates('customer_id').set_index('customer_id')['signup_date'], errors='coerce'
)
campaign_start = pd.to_datetime(
    campaigns.drop_duplicates('campaign_id').set_index('campaign_id')['campaign_date'], errors='coerce'
)
response_dates = pd.to_datetime(responses['response_date'], errors='coerce')
ticket_dates = pd.to_datetime(support_tickets['created_date'], errors='coerce')
transaction_dates = pd.to_datetime(transactions['transaction_date'], errors='coerce')

temporal_quality = pd.DataFrame([
    {'Check': 'Responses before customer signup', 'Invalid Rows': (response_dates < responses['customer_id'].map(customer_signup)).sum()},
    {'Check': 'Responses before campaign date', 'Invalid Rows': (response_dates < responses['campaign_id'].map(campaign_start)).sum()},
    {'Check': 'Tickets before customer signup', 'Invalid Rows': (ticket_dates < support_tickets['customer_id'].map(customer_signup)).sum()},
    {'Check': 'Transactions before customer signup', 'Invalid Rows': (transaction_dates < transactions['customer_id'].map(customer_signup)).sum()},
])
temporal_quality

,Check,Invalid Rows
0,Responses before customer signup,2589
1,Responses before campaign date,6159
2,Tickets before customer signup,1393
3,Transactions before customer signup,2077


# Missing Value Analysis

In [10]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    
    missing = df.isna().sum()
    missing_percent = (missing / len(df) * 100).round(2)
    
    missing_report = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_percent
    })
    
    print(missing_report[missing_report["Missing Count"] > 0])
    print()

Customers
       Missing Count  Missing %
email             40       0.79
phone            166       3.28
city              61       1.21

Campaigns
        Missing Count  Missing %
budget              2       4.88

Responses
             Missing Count  Missing %
customer_id             50       0.33
campaign_id             20       0.13

Support Tickets
                    Missing Count  Missing %
category                       70       0.87
resolution_date              3506      43.55
satisfaction_score           3503      43.52

Transactions
                Missing Count  Missing %
product                    25       0.21
payment_method            130       1.08



# Duplicate Analysis

In [11]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    
    print(f"{name}: {duplicate_count:,} duplicate rows")

Customers: 40 duplicate rows
Campaigns: 1 duplicate rows
Responses: 75 duplicate rows
Support Tickets: 50 duplicate rows
Transactions: 60 duplicate rows


# Customer Data Quality

In [12]:
print("Total customer rows:", len(customers))
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Missing customer IDs:", customers["customer_id"].isna().sum())

Total customer rows: 5060
Unique customer IDs: 5000
Missing customer IDs: 0


In [13]:
email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

invalid_email_mask = (
    customers["email"].notna()
    & ~customers["email"].astype(str).str.match(email_pattern)
)

print("Invalid email records:", invalid_email_mask.sum())

customers.loc[
    invalid_email_mask,
    ["customer_id", "name", "email"]
].head(20)

Invalid email records: 24


,customer_id,name,email
96,C00097,Jannat Hossain,jannat.hossain97@example
438,C00439,Mehedi Akter,mehedi.akter439@example
898,C00899,Jannat Ahmed,jannat.ahmed899@example
1233,C01234,Shakib Chowdhury,shakib.chowdhury1234@example
1327,C01328,Rafi Rahman,rafi.rahman1328@example
1332,C01333,Shakib Hasan,shakib.hasan1333@example
1587,C01588,Karim Rahman,karim.rahman1588@example
1616,C01617,Hossain Khan,hossain.khan1617@example
1881,C01882,Rahim Hossain,rahim.hossain1882@example
1912,C01913,Karim Islam,karim.islam1913@example


In [14]:
print(customers["customer_segment"].value_counts(dropna=False))

customer_segment
Standard    3214
Premium     1418
VIP          347
standard      55
premium       22
vip            4
Name: count, dtype: int64


In [15]:
print(customers["city"].value_counts(dropna=False))

city
Rangpur         658
Chattogram      645
Mymensingh      615
Rajshahi        613
Sylhet          606
Khulna          603
Barishal        580
Dhaka           541
NaN              61
BARISHAL         22
MYMENSINGH       19
KHULNA           18
RANGPUR          14
DHAKA            14
SYLHET           13
RAJSHAHI         12
CHATTOGRAM        6
 Barishal         5
 Sylhet           4
 Mymensingh       4
 Rangpur          2
 Dhaka            2
 Rajshahi         1
 Chattogram       1
 Khulna           1
Name: count, dtype: int64


# Campaign Data Quality

In [16]:
print("Total campaign rows:", len(campaigns))
print("Unique campaign IDs:", campaigns["campaign_id"].nunique())
print("Missing campaign IDs:", campaigns["campaign_id"].isna().sum())

Total campaign rows: 41
Unique campaign IDs: 40
Missing campaign IDs: 0


In [17]:
print("Missing budgets:", campaigns["budget"].isna().sum())
print("Zero budgets:", (campaigns["budget"] == 0).sum())
print("Negative budgets:", (campaigns["budget"] < 0).sum())

Missing budgets: 2
Zero budgets: 0
Negative budgets: 1


In [18]:
campaigns[
    campaigns["budget"].isna() |
    (campaigns["budget"] <= 0)
]

,campaign_id,campaign_name,channel,campaign_date,budget,objective,status,target_audience
13,CMP014,Winter Sale 14,WhatsApp,2026-10-02,NaN,Retention,Scheduled,Standard
19,CMP020,Spring Campaign 20,WhatsApp,2026-01-27,NaN,Engagement,Active,Premium
35,CMP036,Year End Sale 36,Instagram,2026-09-13,-25000.0,Awareness,Active,VIP


In [19]:
campaign_dates = pd.to_datetime(
    campaigns["campaign_date"],
    errors="coerce"
)

print("Invalid campaign dates:", campaign_dates.isna().sum())

Invalid campaign dates: 1


In [20]:
campaigns["channel"].value_counts(dropna=False)

channel
Instagram     9
Google Ads    6
Facebook      6
WhatsApp      6
SMS           5
Email         4
sms           2
google ads    2
email         1
Name: count, dtype: int64

# Response Data Quality

In [21]:
print("Total response rows:", len(responses))
print("Unique response IDs:", responses["response_id"].nunique())
print("Missing response IDs:", responses["response_id"].isna().sum())

Total response rows: 15075
Unique response IDs: 15000
Missing response IDs: 0


In [22]:
responses["response_type"].value_counts(dropna=False)

response_type
Viewed          5905
Clicked         4162
Responded       2355
Purchase        1986
Unsubscribed     516
viewed            68
clicked           33
responded         30
purchase          12
unsubscribed       8
Name: count, dtype: int64

In [23]:
print(
    responses["converted"]
    .value_counts(dropna=False)
)

converted
0    13059
1     1996
2       20
Name: count, dtype: int64


In [24]:
invalid_converted = ~responses["converted"].isin([0, 1])

print(
    "Invalid converted values:",
    invalid_converted.sum()
)

responses.loc[
    invalid_converted,
    ["response_id", "customer_id", "campaign_id", "converted"]
].head(20)

Invalid converted values: 20


,response_id,customer_id,campaign_id,converted
1131,R001132,C01438,CMP007,2
2268,R002269,C02853,CMP007,2
3458,R003459,C04749,CMP016,2
3640,R003641,C04015,CMP017,2
4092,R004093,C03083,CMP033,2
4116,R004117,C00848,CMP037,2
4375,R004376,C03253,CMP027,2
4544,R004545,C02464,CMP015,2
4788,R004789,C03648,CMP013,2
5582,R005583,C04637,CMP032,2


In [25]:
response_dates = pd.to_datetime(
    responses["response_date"],
    errors="coerce"
)

print("Invalid response dates:", response_dates.isna().sum())

Invalid response dates: 15


# Support Ticket Quality

In [26]:
print("Total tickets:", len(support_tickets))
print("Unique ticket IDs:", support_tickets["ticket_id"].nunique())
print("Missing ticket IDs:", support_tickets["ticket_id"].isna().sum())

Total tickets: 8050
Unique ticket IDs: 8000
Missing ticket IDs: 0


In [27]:
support_tickets["category"].value_counts(dropna=False)

category
Delivery        1338
Payment         1284
Product         1115
Technical        967
Account          762
Refund           762
Order            667
Subscription     558
Other            347
NaN               70
payment           37
product           26
delivery          25
refund            24
order             21
account           15
technical         12
subscription      10
other             10
Name: count, dtype: int64

In [28]:
support_tickets["priority"].value_counts(dropna=False)

priority
Medium        3915
High          1946
Low           1625
Critical       474
 Medium         49
 High           27
 Low            10
 Critical        4
Name: count, dtype: int64

In [29]:
support_tickets["status"].value_counts(dropna=False)

status
Resolved        3016
Closed          1472
In Progress     1445
Open            1226
Pending          801
Resolved          35
In Progress       16
Open              14
Pending           13
Closed            12
Name: count, dtype: int64

In [30]:
invalid_satisfaction = (
    support_tickets["satisfaction_score"].notna()
    & ~support_tickets["satisfaction_score"].between(1, 5)
)

print(
    "Invalid satisfaction scores:",
    invalid_satisfaction.sum()
)

support_tickets.loc[
    invalid_satisfaction,
    ["ticket_id", "customer_id", "satisfaction_score"]
].head(20)

Invalid satisfaction scores: 25


,ticket_id,customer_id,satisfaction_score
61,T000062,C01421,10.0
127,T000128,C01390,10.0
614,T000615,C04710,10.0
808,T000809,C00290,10.0
817,T000818,C03896,10.0
933,T000934,C02578,10.0
997,T000998,C03695,10.0
1269,T001270,C00943,10.0
1773,T001774,C00587,10.0
1897,T001898,C00580,10.0


In [31]:
created_dates = pd.to_datetime(
    support_tickets["created_date"],
    errors="coerce"
)

resolution_dates = pd.to_datetime(
    support_tickets["resolution_date"],
    errors="coerce"
)

invalid_resolution_order = (
    resolution_dates.notna()
    & created_dates.notna()
    & (resolution_dates < created_dates)
)

print(
    "Resolution dates before creation dates:",
    invalid_resolution_order.sum()
)

Resolution dates before creation dates: 15


# Transaction Data Quality

In [32]:
print("Total transactions:", len(transactions))
print("Unique transaction IDs:", transactions["transaction_id"].nunique())
print("Missing transaction IDs:", transactions["transaction_id"].isna().sum())

Total transactions: 12060
Unique transaction IDs: 12000
Missing transaction IDs: 0


In [33]:
print("Missing amounts:", transactions["amount"].isna().sum())
print("Zero amounts:", (transactions["amount"] == 0).sum())
print("Negative amounts:", (transactions["amount"] < 0).sum())

Missing amounts: 0
Zero amounts: 20
Negative amounts: 30


In [34]:
transactions[
    transactions["amount"].isna() |
    (transactions["amount"] <= 0)
].head(20)

,transaction_id,customer_id,transaction_date,product,amount,payment_method,status
56,TRX000057,C00423,2026-08-07,Data Insights Package,-12117.02,Nagad,Refunded
293,TRX000294,C02696,2026-09-19,Premium Subscription,-3069.90,Nagad,Completed
671,TRX000672,C04552,2026-02-23,Business Plan,0.00,Bkash,Completed
681,TRX000682,C02170,2026-08-16,Enterprise Plan,-43933.48,Bkash,Pending
813,TRX000814,C03195,2026-03-02,Enterprise Plan,-56601.13,Nagad,Completed
2088,TRX002089,C03420,2026-08-24,Marketing Package,-11885.95,Card,Completed
2340,TRX002341,C03116,2026-09-09,Cloud Storage,-3044.14,Nagad,Failed
2593,TRX002594,C02387,2026-07-17,Cloud Storage,0.00,Nagad,Completed
2625,TRX002626,C04134,2026-08-09,Standard Subscription,0.00,Bank Transfer,Failed
2838,TRX002839,C02477,2026-01-31,Premium Subscription,-2149.79,PayPal,Completed


In [35]:
transactions["status"].value_counts(dropna=False)

status
Completed    10163
Pending        818
Refunded       572
Failed         487
Unknown         20
Name: count, dtype: int64

# Relationship / Foreign-Key Validation

# Response → Customer relationship

In [36]:
valid_customer_ids = set(
    customers["customer_id"].dropna()
)

invalid_response_customers = (
    responses["customer_id"].notna()
    & ~responses["customer_id"].isin(valid_customer_ids)
)

print(
    "Responses with invalid customer IDs:",
    invalid_response_customers.sum()
)

Responses with invalid customer IDs: 8


# Response → Campaign relationship

In [37]:
valid_campaign_ids = set(
    campaigns["campaign_id"].dropna()
)

invalid_response_campaigns = (
    responses["campaign_id"].notna()
    & ~responses["campaign_id"].isin(valid_campaign_ids)
)

print(
    "Responses with invalid campaign IDs:",
    invalid_response_campaigns.sum()
)

Responses with invalid campaign IDs: 7


# Support → Customer relationship

In [38]:
invalid_ticket_customers = (
    support_tickets["customer_id"].notna()
    & ~support_tickets["customer_id"].isin(valid_customer_ids)
)

print(
    "Tickets with invalid customer IDs:",
    invalid_ticket_customers.sum()
)

Tickets with invalid customer IDs: 15


# Transactions → Customer relationship

In [39]:
invalid_transaction_customers = (
    transactions["customer_id"].notna()
    & ~transactions["customer_id"].isin(valid_customer_ids)
)

print(
    "Transactions with invalid customer IDs:",
    invalid_transaction_customers.sum()
)

Transactions with invalid customer IDs: 20


# Numerical Summary

In [40]:
print("Campaign Budget Statistics")
display(campaigns["budget"].describe())

Campaign Budget Statistics


count        39.000000
mean      63461.538462
std       56208.655521
min      -25000.000000
25%       15000.000000
50%       40000.000000
75%      125000.000000
max      150000.000000
Name: budget, dtype: float64

In [41]:
print("Transaction Amount Statistics")
display(transactions["amount"].describe())

Transaction Amount Statistics


count    12060.000000
mean     13943.727218
std      18961.823687
min     -70793.050000
25%       3017.332500
50%       6321.555000
75%      16449.485000
max      99995.280000
Name: amount, dtype: float64

In [42]:
print("Customer Satisfaction Statistics")
display(support_tickets["satisfaction_score"].describe())

Customer Satisfaction Statistics


count    4547.000000
mean        3.952056
std         1.111428
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        10.000000
Name: satisfaction_score, dtype: float64

# Create a Final Data Quality Report

In [43]:
quality_report = []

for name, df in datasets.items():

    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isna().sum().sum()

    quality_report.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Duplicate Rows": df.duplicated().sum(),
        "Missing Cells": missing_cells,
        "Missing %": round(
            (missing_cells / total_cells) * 100, 2
        )
    })

quality_report_df = pd.DataFrame(quality_report)

quality_report_df

,Dataset,Rows,Columns,Duplicate Rows,Missing Cells,Missing %
0,Customers,5060,7,40,267,0.75
1,Campaigns,41,8,1,2,0.61
2,Responses,15075,6,75,70,0.08
3,Support Tickets,8050,8,50,7079,10.99
4,Transactions,12060,7,60,155,0.18


# Save the Quality Report

In [44]:
quality_report_df.to_csv(
    "../data/processed/raw_quality_report.csv",
    index=False
)

print("Quality report saved successfully.")

Quality report saved successfully.
